# Vehicle listing EDA

Run this notebook from the **repository root**.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import load_config, load_vehicle_dataframe

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)

cfg = load_config(ROOT / "config" / "config.yaml")
df = load_vehicle_dataframe(cfg)
df.head()

## Price distribution

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df["selling_price"], kde=True, ax=ax, color="#2563eb")
ax.set_title("Selling price distribution")
ax.set_xlabel("selling_price")
plt.tight_layout()

## Brand vs price (top 15 brands by median)

In [ ]:
brand_order = (
    df.groupby("brand")["selling_price"]
    .median()
    .sort_values(ascending=False)
    .head(15)
    .index.tolist()
)
subset = df[df["brand"].isin(brand_order)]
fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(data=subset, x="brand", y="selling_price", order=brand_order, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_title("Brand vs selling price")
plt.tight_layout()

## km_driven vs selling_price scatter

In [ ]:
sample = df.sample(n=min(5000, len(df)), random_state=42)
fig, ax = plt.subplots()
sns.scatterplot(
    data=sample,
    x="km_driven",
    y="selling_price",
    hue="fuel_type",
    alpha=0.35,
    ax=ax,
)
ax.set_title("km_driven vs selling_price")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

## Correlation heatmap

In [ ]:
numeric_cols = ["year", "km_driven", "selling_price"]
corr = df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="vlag", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation heatmap")
plt.tight_layout()

## Year vs average price trend

In [ ]:
yearly = df.groupby("year", as_index=False)["selling_price"].mean()
yearly.rename(columns={"selling_price": "avg_price"}, inplace=True)
fig, ax = plt.subplots()
sns.lineplot(data=yearly, x="year", y="avg_price", marker="o", ax=ax, color="#1d4ed8")
ax.set_title("Mean selling price by year")
ax.set_xlabel("year")
ax.set_ylabel("mean selling_price")
plt.tight_layout()